# 1. Imports and Setup
This section initializes the required libraries and modules.

In [1]:

# Standard Library Imports
import os
import logging

# Third-Party Library Imports
import numpy as np

# Project-Specific Imports
from utils.logging_config import setup_logging
from utils.model_utils import train_or_load_model
from utils.utils import (
    Compression,
    initialize_directories,
    save_results,
)
from utils.calibration import (
    calibrate_geometric,
    calibrate_parametric,
    calibrate_trust_score,
    compute_uncalibrated_metrics,
)
from utils.data_utils import (
    transform_test_set,
    load_and_split_data,
)
from calibrators.calibrators import (
    IsotonicCalibrator,
    PlattCalibrator,
    SBCCalibrator,
    HBCalibrator,
    BBQCalibrator,
    BetaCalibrator,
    EnsembleTSCalibrator,
)
from calibrators.non_parametric_calibrators import IsotonicCalibrator
from calibrators.parametric_calibrators import TemperatureScalingCalibrator

# Initialize logging
setup_logging()
logger = logging.getLogger(__name__)


2025-01-24 18:30:45,399 - INFO - Loading faiss with AVX2 support.
2025-01-24 18:30:45,421 - INFO - Successfully loaded faiss with AVX2 support.


# 2. Helper Functions
These functions handle dataset preparation and model setup.

In [2]:
def prepare_data(dataset_name, random_state, transformed):
    """Load and optionally transform dataset."""
    X_train, X_val, X_test, y_train, y_val, y_test = load_and_split_data(dataset_name, random_state)
    if transformed:
        X_test = transform_test_set(X_test)
    return X_train, X_val, X_test, y_train, y_val, y_test


def prepare_model(X_train, y_train, X_val, y_val, dataset_name, random_state, model_type, dataset_epochs):
    """Train or load the model."""
    return train_or_load_model(
        X_train, y_train, X_val, y_val,
        dataset_name=dataset_name,
        random_state=random_state,
        model_type=model_type,
        epochs_dict=dataset_epochs
    )


# 3. Main Function
Orchestrates the entire workflow for geometric calibration.

In [3]:
from scripts.main import perform_calibrations


def main(dataset_name, random_state, model_type="cnn", metric="L2", transformed=False, trust_alpha=0.1,
         compression_types=None, compression_params=2):
    base_path = f"output/{dataset_name}/{random_state}/{model_type}/{metric}/{compression_types}/{compression_params}"
    if transformed:
        base_path = os.path.join(base_path, "transformed")
    all_results_path = os.path.join(base_path, "all")
    for _, _, files in os.walk(all_results_path):
        for file in files:
            if file.endswith("all_results.csv"):
                logger.info(f"Results file found at {os.path.join(all_results_path, file)}.")
                return

    # Initialize directories
    base_dir, technique_dirs, all_results_dir = initialize_directories(
        f"output/{dataset_name}/{random_state}/{model_type}/{metric}/{compression_types}/{compression_params}",
        transformed, dataset_name, random_state, model_type, metric, trust_alpha
    )
    compression = None
    if compression_types:
        logger.info(f"Initializing Compression with types: {compression_types}, params: {compression_params}")
        compression = Compression(compression_types=compression_types, compression_params=compression_params)

    # Load data
    X_train, X_val, X_test, y_train, y_val, y_test = prepare_data(dataset_name, random_state, transformed)

    # Train or load model
    dataset_epochs = {
        "mnist": 10,
        "fashion_mnist": 20,
        "cifar10": 35,
        "cifar100": 75,
        "sign_language": 25,
        "gtsrb": 20,
    }
    model = prepare_model(X_train, y_train, X_val, y_val, dataset_name, random_state, model_type, dataset_epochs)

    # Get dataset sizes
    train_size, val_size, test_size = len(X_train), len(X_val), len(X_test)

    # Preprocess data based on the model type
    if model_type in ["cnn", "densenet", "pretrained_resnet", "pretrained_efficientnet"]:
        logger.info(f"Using TensorFlow/Keras model: {model_type}")
        features_test = model.predict(X_test)
        y_test_pred = np.argmax(features_test, axis=1)
        features_val = model.predict(X_val)
        y_val_pred = np.argmax(features_val, axis=1)
    else:
        logger.info(f"Using sklearn model: {model_type}")
        X_test = X_test.reshape(X_test.shape[0], -1) if len(X_test.shape) > 2 else X_test
        X_val = X_val.reshape(X_val.shape[0], -1) if len(X_val.shape) > 2 else X_val

        features_test = model.predict_proba(X_test)
        y_test_pred = model.predict(X_test)
        features_val = model.predict_proba(X_val)
        y_val_pred = model.predict(X_val)

    logger.info(f"Feature extraction complete for model type: {model_type}")
    results = []
    uncalibrated_results = compute_uncalibrated_metrics(features_test, y_test_pred, y_test, train_size, val_size, test_size)
    results.append(uncalibrated_results)

    calibrations = {
    "isotonic": IsotonicCalibrator(),
    "platt": PlattCalibrator(),
    "temperature": TemperatureScalingCalibrator(),
    "sbc": SBCCalibrator(bins=15),  # Using SBCCalibrator with 15 bins
    "hb": HBCalibrator(bins=50),    # Using HBCalibrator with 50 bins
    "bbq": BBQCalibrator(bins=50),  # Using BBQCalibrator with 50 bins
    "beta": BetaCalibrator(bins=50), # Using BetaCalibrator with 50 bins
    "ets": EnsembleTSCalibrator(temperature=1.0),
    "fast_separation": {"library": "fast_separation", "mode": None},  # Fast Separation
    "separation": {"library": "separation", "mode": None},  # Regular Separation
    }

    results, existing_results = perform_calibrations(
        calibrations, model, X_train, y_train, X_val, y_val, X_test, y_test,
        features_val, features_test, y_test_pred, technique_dirs, trust_alpha,
        metric, compression
    )
    save_results(results, existing_results, all_results_dir)


# 4. Execution
Running the main function interactively.

In [ ]:
datasets = ["GTSRB","CIFAR100"]
models = ["RF", "GB"]  # Add more models if needed
metrics = ["L2"]  # Add more metrics if needed
compression_configs = [ # Compression configurations
    (["None"], [0]),
    (["Avgpool"], [2]),
    (["Maxpool"], [4])
]

# Iteration over datasets, models, metrics, and compression configurations
for dataset in datasets:
    print(f"Running for dataset: {dataset}")

    for shuffle_num in range(311, 312):  # Example shuffle range
        for metric in metrics:
            for model in models:
                for compression_types, compression_params in compression_configs:
                    # Print configuration for each iteration
                    print(f"Dataset: {dataset}, Model: {model}, Shuffle: {shuffle_num}, Metric: {metric}, Compression: {compression_types}")

                    # Execute the main function with the current configuration
                    main(
                        dataset_name=dataset,
                        random_state=shuffle_num,
                        model_type=model,
                        metric=metric,
                        transformed=False,  # Adjust as needed
                        compression_types=compression_types,  # Adjust as needed
                        compression_params=compression_params  # Adjust as needed
                    )



2025-01-24 18:30:47,026 - INFO - Results file found at output/GTSRB/311/RF/L2/['None']/[0]\all\all_results.csv.
2025-01-24 18:30:47,029 - INFO - Initializing Compression with types: ['Avgpool'], params: [2]
2025-01-24 18:30:47,030 - INFO - Initialize Compression with ['Avgpool'] and [2]
2025-01-24 18:30:47,031 - INFO - Loading dataset: GTSRB
2025-01-24 18:30:47,063 - INFO - Loading training images...


Running for dataset: GTSRB
Dataset: GTSRB, Model: RF, Shuffle: 311, Metric: L2, Compression: ['None']
Dataset: GTSRB, Model: RF, Shuffle: 311, Metric: L2, Compression: ['Avgpool']


2025-01-24 18:31:00,661 - INFO - Loading test images...
2025-01-24 18:31:05,326 - INFO - Loaded 51839 images with shape (32, 32, 3)
2025-01-24 18:31:05,599 - INFO - Train shape: (31103, 32, 32, 3), Validation shape: (10368, 32, 32, 3), Test shape: (10368, 32, 32, 3)
2025-01-24 18:31:05,623 - INFO - Loading pre-trained RF model from output/GTSRB/311/saved_models\RF_model.keras.
2025-01-24 18:31:06,025 - INFO - Using sklearn model: RF
2025-01-24 18:31:10,016 - INFO - Feature extraction complete for model type: RF
2025-01-24 18:31:10,016 - INFO - Starting Expected Calibration Error (ECE) calculation.
2025-01-24 18:31:10,018 - INFO - Final ECE value: 0.3408960334785349
2025-01-24 18:31:10,019 - INFO - Calculating Maximum Calibration Error.
2025-01-24 18:31:10,023 - INFO - Final MCE value: 0.6575613343167042
2025-01-24 18:31:10,024 - INFO - Initialized IsotonicCalibrator with n_classes=None, bins=15, temperature=1.0
2025-01-24 18:31:10,024 - INFO - Initialized PlattCalibrator with n_classes

Uncalibrated Metrics: {'ECE': 0.3408960334785349, 'MCE': 0.6575613343167042}


2025-01-24 18:31:10,628 - INFO - SBCCalibrator: Successfully fitted the model.
2025-01-24 18:31:10,630 - INFO - Calibrating test set with sbc.
2025-01-24 18:31:10,630 - INFO - SBCCalibrator: Calibrating probabilities.
2025-01-24 18:31:10,650 - INFO - Shape of calibrated_probs: (10368, 43)
2025-01-24 18:31:10,651 - INFO - Length of y_test: 10368
2025-01-24 18:31:10,651 - INFO - Shape of features_test: (10368, 43)
2025-01-24 18:31:10,653 - INFO - Calculating metrics for sbc.
2025-01-24 18:31:10,654 - INFO - Starting Expected Calibration Error (ECE) calculation.
2025-01-24 18:31:10,656 - INFO - Final ECE value: 0.1554975394978959
2025-01-24 18:31:10,657 - INFO - Calculating Maximum Calibration Error.
2025-01-24 18:31:10,659 - INFO - Final MCE value: 0.8141585700935619
2025-01-24 18:31:10,663 - INFO - Sbc Metrics saved to output/GTSRB/311/RF/L2/['Avgpool']/[2]\sbc\results.csv.
2025-01-24 18:31:10,664 - INFO - Found existing results for hb. Adding to final results.
2025-01-24 18:31:10,666 -